<a href="https://colab.research.google.com/github/ynam0327-afk/REDRED/blob/main/smishing_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
!rm -rf /content/REDRED
!git clone https://github.com/ynam0327-afk/REDRED.git
!jupyter nbconvert --to script /content/REDRED/content_authenticity.ipynb
!jupyter nbconvert --to script /content/REDRED/text_match.ipynb
!jupyter nbconvert --to script /content/REDRED/model.ipynb
!jupyter nbconvert --to script /content/REDRED/official_sms_check.ipynb
import sys
sys.path.insert(0, "/content/REDRED")

import os
for name in ["content_authenticity", "text_match", "model", "official_sms_check"]:
    txt_path = f"/content/REDRED/{name}.txt"
    py_path = f"/content/REDRED/{name}.py"
    if os.path.exists(txt_path) and not os.path.exists(py_path):
        os.rename(txt_path, py_path)

import pandas as pd
from datetime import date as date_cls
from urllib.parse import urlparse
from google.colab import drive
drive.mount('/content/drive')

from content_authenticity import content_authenticity_score
from text_match import match_sms_to_call119
from model import url_risk_score_model, is_official_domain, load_url_model
from official_sms_check import official_sms_reliability
from cross_validation import SourceMatch, combined_disaster_reliability


# ---------------------------------------------------------------------------
# 모듈 로드 시 딱 한 번만 실행 - 무거운 것들을 여기서 메모리에 올려둔다
# ---------------------------------------------------------------------------

_RF_MODEL = load_url_model("/content/drive/MyDrive/REDRED/rf_url_model.joblib")

_FIRE_DB = pd.read_csv("/content/drive/MyDrive/REDRED/fire_events_region_normalized.csv")
_FIRE_DB["report_date"] = _FIRE_DB["report_date"].astype(str)

_CALL119_BY_CITY = {
    "서울특별시": pd.read_parquet("/content/drive/MyDrive/REDRED/seoul_119_2024_dates.parquet"),
    "부산광역시": pd.read_csv("/content/drive/MyDrive/REDRED/busan_119_2024_dates.csv"),
}
for _df in _CALL119_BY_CITY.values():
     _df["dclr_ymd"] = _df["dclr_ymd"].astype(str)


# ---------------------------------------------------------------------------
# 개별 소스 조회 함수
# ---------------------------------------------------------------------------

def _check_fire_db(region: str, report_date: str) -> SourceMatch:
    if not region:
        return SourceMatch(False)
    sido = region.split()[0] if region.split() else None
    cand = _FIRE_DB[(_FIRE_DB["report_date"] == report_date) & (_FIRE_DB["sido_official"] == sido)]
    if len(cand) == 0:
        return SourceMatch(False)
    return SourceMatch(True, confidence=0.6)


def _check_call119(message: str, region: str, sms_date: str) -> SourceMatch:
    sido = region.split()[0] if region and region.split() else None
    call119_df = _CALL119_BY_CITY.get(sido)
    if call119_df is None:
        return None  # 서울/부산 외 지역 - 애초에 커버리지 없음(정보부족 처리 대상)

    row = pd.Series({"date": sms_date, "region": region, "message": message,
                      "disaster_type": None})
    result = match_sms_to_call119(row, call119_df)
    if result is None:
        return SourceMatch(False)

    hour_diff = result.get("hour_diff")
    confidence = max(0.0, 1 - hour_diff / 12) if hour_diff is not None else 0.5
    return SourceMatch(True, confidence=confidence)


def _extract_domain(url: str) -> str:
    if not url:
        return ""
    if not url.startswith(("http://", "https://")):
        url = "http://" + url
    return urlparse(url).netloc.lower()


# ---------------------------------------------------------------------------
# 외부(ingest-worker 등)에서 부르는 단일 진입점
# ---------------------------------------------------------------------------

def process_message(raw_text: str, url: str | None, region: str | None,
                     sms_date: str | None = None,
                     official_service_key: str | None = None) -> dict:
    """
    반환: {"url_risk_score": float, "text_authenticity_score": float, "detail": {...}}
    /messages에는 url_risk_score, text_authenticity_score 두 값만 넘기면 된다.
    """
    sms_date = sms_date or date_cls.today().isoformat()
    ymd = sms_date.replace("-", "")

    # 1) URL 위험도 - model.py의 RF 모델 사용 (domain_module의 규칙 기반은 안 씀)
    domain = _extract_domain(url)
    is_wl = bool(url) and is_official_domain(domain)
    url_risk_score = 0.0 if (not url or is_wl) else url_risk_score_model(url, _RF_MODEL)

    # 2) 재난정보 신뢰도 - 공식API > (소방청DB + 119신고접수 결합) > 콘텐츠 판단 순
    official_score = None
    if official_service_key:
        try:
            official_result = official_sms_reliability(raw_text, ymd, region)
            official_score = official_result.get("score")
        except Exception:
            official_score = None  # API 장애 시에도 파이프라인은 계속 진행

    fire_db_match = _check_fire_db(region, sms_date)
    call119_match = _check_call119(raw_text, region, sms_date)

    disaster = combined_disaster_reliability(
        fire_db_match, call119_match, raw_text, official_score
    )

    return {
        "url_risk_score": round(url_risk_score, 4),
        "text_authenticity_score": disaster["score"],
        "detail": {
            "url_whitelisted": is_wl,
            "disaster_matched_sources": disaster["matched_sources"],
            "disaster_note": disaster["note"],
        },
    }


# ---------------------------------------------------------------------------
# 검증
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    result = process_message(
        raw_text="오늘 09:34 원주시 원동 한주아파트 101동 건물에서 화재 발생. 차량은 건물 주변 도로를 우회하고, 건물 내 시민은 건물 밖으로 대피하세요. [원주시]",
        url=None,
        region="강원특별자치도 원주시",
        sms_date="2024-06-03",
    )
    print(result)


fatal: destination path 'REDRED' already exists and is not an empty directory.
Traceback (most recent call last):
  File "/usr/local/bin/jupyter-nbconvert", line 4, in <module>
  File "/usr/local/lib/python3.12/dist-packages/nbconvert/nbconvertapp.py", line 193, in <module>
    class NbConvertApp(JupyterApp):
  File "/usr/local/lib/python3.12/dist-packages/nbconvert/nbconvertapp.py", line 252, in NbConvertApp
    Options include {get_export_names()}.
                     ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/nbconvert/exporters/base.py", line 152, in get_export_names
    e = get_exporter(exporter_name)(config=config)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/nbconvert/exporters/base.py", line 107, in get_exporter
    exporters = entry_points(group="nbconvert.exporters")
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/metadata/__init__.py", line 913, in entry_points
   